In [4]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!unzip "/content/drive/MyDrive/DS.zip" -d /content/

In [ ]:
import os
import shutil
import random

source = "/content/DS"

target = "/content/dataset"

classes = [
    "cnc",
    "cnc_anomoly",
    "washing_machine",
    "washing_machine_anomoly"
]

for split in ["train","val","test"]:
    for cls in classes:
        os.makedirs(
            os.path.join(target,split,cls),
            exist_ok=True
        )

for cls in classes:

    path = os.path.join(source,cls)

    images = [
        f for f in os.listdir(path)
        if f.endswith((".jpg",".jpeg",".png"))
    ]

    random.shuffle(images)

    train_end = int(0.8*len(images))
    val_end = int(0.9*len(images))

    train = images[:train_end]
    val = images[train_end:val_end]
    test = images[val_end:]

    for img in train:
        shutil.copy(
            os.path.join(path,img),
            os.path.join(target,"train",cls,img)
        )

    for img in val:
        shutil.copy(
            os.path.join(path,img),
            os.path.join(target,"val",cls,img)
        )

    for img in test:
        shutil.copy(
            os.path.join(path,img),
            os.path.join(target,"test",cls,img)
        )

print("Done")

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-cls.pt")

model.train(
    data="/content/dataset",
    epochs=50,
    imgsz=224,
    batch=16
)

In [ ]:
from ultralytics import YOLO
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import os

model = YOLO(
"/content/runs/classify/train/weights/best.pt"
)

y_true=[]
y_pred=[]

test_dir="/content/dataset/test"

for cls in os.listdir(test_dir):

    cls_path=os.path.join(test_dir,cls)

    if not os.path.isdir(cls_path):
        continue

    for img in os.listdir(cls_path):

        if not img.endswith(
            (".jpg",".jpeg",".png")
        ):
            continue

        img_path=os.path.join(
            cls_path,
            img
        )

        result=model(img_path)

        pred=result[0].names[
            result[0].probs.top1
        ]

        y_true.append(cls)
        y_pred.append(pred)

accuracy=accuracy_score(
    y_true,
    y_pred
)

precision=precision_score(
    y_true,
    y_pred,
    average="weighted"
)

recall=recall_score(
    y_true,
    y_pred,
    average="weighted"
)

f1=f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print("Accuracy :",accuracy)
print("Precision:",precision)
print("Recall   :",recall)
print("F1 Score :",f1)

In [ ]:
from ultralytics import YOLO
import cv2
from collections import Counter
import os

# Load trained model
model = YOLO("/content/runs/classify/train/weights/best.pt")

# Input video - User will provide the path
print("Please list files in your Google Drive to find your video (e.g., !ls /content/drive/MyDrive/your_folder):")
video_path = input("Enter the full path to your video file (e.g., /content/drive/MyDrive/my_video.mp4): ")

cap = cv2.VideoCapture(video_path)

predictions = []

frame_count = 0

if not cap.isOpened():
    print(f"Error: Could not open video file at {video_path}")
else:
    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_count += 1

        # Analyze 1 frame every 10 frames
        if frame_count % 10 != 0:
            continue

        results = model(frame)

        pred_class = results[0].names[
            results[0].probs.top1
        ]

        predictions.append(pred_class)

    cap.release()

# Count predictions
counts = Counter(predictions)

print("\nFrame Analysis")
print(counts)

# Majority voting
if predictions:
    final_class = counts.most_common(1)[0][0]

    total = len(predictions)

    confidence = (
        counts[final_class] / total
    ) * 100

    print("\nFinal Result")
    print("Machine Status :", final_class)
    print("Confidence     :", round(confidence,2), "%")
else:
    print("\nNo frames were processed. Please check the video path and content.")

In [ ]:
from google.colab import files

uploaded = files.upload()

video_path = list(uploaded.keys())[0]

print("Selected Video:", video_path)

In [ ]:
from google.colab import files
from ultralytics import YOLO
from collections import Counter
import cv2

# Upload video from local PC
uploaded = files.upload()

video_path = list(uploaded.keys())[0]

# Load model
model = YOLO("/content/runs/classify/train/weights/best.pt")

cap = cv2.VideoCapture(video_path)

predictions = []

frame_count = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_count += 1

    # Process every 10th frame
    if frame_count % 10 != 0:
        continue

    result = model(frame)

    pred = result[0].names[
        result[0].probs.top1
    ]

    predictions.append(pred)

cap.release()

counts = Counter(predictions)

total = len(predictions)

normal_count = (
    counts.get("cnc", 0)
    + counts.get("washing_machine", 0)
)

anomaly_count = (
    counts.get("cnc_anomoly", 0)
    + counts.get("washing_machine_anomoly", 0)
)

health_score = (normal_count / total) * 100 if total > 0 else 0

print("\n========== MACHINE HEALTH REPORT ==========")
print("Healthy Frames :", normal_count)
print("Anomaly Frames :", anomaly_count)
print("Health Score   :", round(health_score, 2), "%")

if health_score >= 80:
    print("Status : HEALTHY")
elif health_score >= 60:
    print("Status : WARNING")
else:
    print("Status : HIGH RISK")

print("==========================================")